# **Mid-Term Project**

In [1]:
%pip install -q \
    langchain==0.3.25 \
    langchain-community==0.3.24 \
    langchain-openai==0.3.17 \
    chromadb==0.5.23 \
    openai \
    tiktoken \
    opentelemetry-api==1.24.0 \
    opentelemetry-sdk==1.24.0 \
    opentelemetry-semantic-conventions==0.45b0

Note: you may need to restart the kernel to use updated packages.


### **Mount Google Drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

### **Text Scraping**


In [2]:
import os
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/",
    "https://datascience.uchicago.edu/education/masters-programs/in-person-program/",
    "https://datascience.uchicago.edu/education/masters-programs/online-program/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-project-archive/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines/",
    "https://datascience.uchicago.edu/education/tuition-fees-aid/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/our-students/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/instructors-staff/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/",
    "https://datascience.uchicago.edu/explore-the-ms-ads-campus/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/career-outcomes/",
]

output_dir = "/content/drive/MyDrive/GenAI/Midterm Project/texts"
os.makedirs(output_dir, exist_ok=True)

loader = WebBaseLoader(urls)
docs = loader.load()

for i, doc in enumerate(docs):
    url = doc.metadata.get("source", "")
    name = url.rstrip("/").split("/")[-1]

    if name == "":
        name = "overview"

    filename = f"{name}.txt"
    filepath = os.path.join(output_dir, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"Title: {doc.metadata.get('title', 'Untitled')}\n")
        f.write(f"URL: {url}\n\n")
        f.write(doc.page_content)

    print(f"Saved: {filepath}")

print(f"\nDone. Loaded {len(docs)} pages.")


USER_AGENT environment variable not set, consider setting it to identify your requests.


OSError: [Errno 30] Read-only file system: '/content'

### Loading Documents

In [18]:
!pip install langchain langchain-community
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# Path to the airline knowledge base
UchiADSP = "/content/drive/MyDrive/GenAI/Midterm Project/texts"


# Load all .txt files in the directory
loader = DirectoryLoader(
    UchiADSP,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
raw_docs = loader.load()

totalcharacters = sum(len(doc.page_content) for doc in raw_docs)

print(f"Loaded {len(raw_docs)} documents")

print(f"Total characters: {totalcharacters:,}")

for doc in raw_docs:
    source = doc.metadata.get('source', 'unknown')
    print(f"  {source.split('/')[-1]:40s}  {len(doc.page_content):6,} chars")

Loaded 14 documents
Total characters: 287,124
  ms-in-applied-data-science.txt             9,283 chars
  in-person-program.txt                     38,165 chars
  online-program.txt                        38,094 chars
  capstone-projects.txt                     10,603 chars
  capstone-project-archive.txt               8,062 chars
  course-progressions.txt                   42,967 chars
  how-to-apply.txt                          13,185 chars
  events-deadlines.txt                      13,395 chars
  tuition-fees-aid.txt                       9,506 chars
  our-students.txt                           8,419 chars
  instructors-staff.txt                     57,541 chars
  faqs.txt                                  22,498 chars
  explore-the-ms-ads-campus.txt              8,039 chars
  career-outcomes.txt                        7,367 chars


In [19]:
print("--- Sample document content ---")
print(raw_docs[0].metadata['source'].split('/')[-1])
print(raw_docs[0].page_content[:500])
print("...")

--- Sample document content ---
ms-in-applied-data-science.txt
Title: Master's in Applied Data Science | DSI
URL: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/

    Master's in Applied Data Science | DSI               Skip to main content                  About  About the Data Science InstituteThe Data Science Institute (DSI) executes the University of Chicago’s bold, innovative vision of Data Science as a new discipline. Jobs & OpportunitiesOpen faculty, postdoctoral, staff, and student roles with the UChicago Data
...


## **Chunking**

This next step is chunking the text data that has been loaded using LangChain

In [20]:
# If needed:
# !pip install -qU langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)

chunks = splitter.split_documents(raw_docs)

print(f"Created {len(chunks)} chunks\n")

sample_idx = min(5, len(chunks) - 1)
sampletext = chunks[sample_idx].page_content[:200]
print(f'Sample chunk (first 200 chars): "{sampletext}"')

source = chunks[sample_idx].metadata.get("source", "unknown")
print(f"Chunk source: {source}")

Created 730 chunks

Sample chunk (first 200 chars): "AICE: AI for ClimateInter-discplinary integration of AI with fundamental domain knowledge to accelerate and transform climate research with a focus on both scientific advances and societal impacts. Da"
Chunk source: /content/drive/MyDrive/GenAI/Midterm Project/texts/ms-in-applied-data-science.txt


## **Embeddings and Vector Store**

In [21]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key loaded:", os.environ["OPENAI_API_KEY"][:8], "...")

API key loaded: sk-proj- ...


In [22]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)

# Build the vector store (this calls the OpenAI API for each chunk)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="UChi_Midterm"
)

print(f"Vector store contains  {vectorstore._collection.count()} vectors")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

### Verifying Retrieval

In [ ]:
query = "What are the career outcomes of MS in Applied Data Science graduates?"

docs_and_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: {query}")
print("\nTop 3 retrieved chunks:\n")

for i, (doc, score) in enumerate(docs_and_scores, 1):
    source = doc.metadata["source"].split("/")[-1]
    print(f"[{i}] Source: {source} | Score: {score:.4f}")
    print(doc.page_content[:500])
    print("-" * 80)

### **Setting up Rag Chain**

In this section I"m setting up the Retreiver. We are building a multi query retrieval system.

Multi query retrieval is implmented to reduce our vector search missing relevant information due to wording differences. This improves our chances of capturing relevant information across pages that use slightly different language (“career outcomes” vs. “job placement” vs. “employment results”).

We are also implementing a rag fusion technique using recirprocal rag fusion. After retrieving document chunks for each rewritten query, the system essentially reranks the rsults based on how highly and how consistenly they appear across multiple retreiver lists.

ultimately, the top chunks will be passed into the LLM for answering.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
import json

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# Generate multiple query variations from the original question
prompt_multi = ChatPromptTemplate.from_template(
    """Generate 4 diverse search queries for retrieving documents that answer the user's question.
Include paraphrases, synonyms, and related terminology that may appear on a webpage.

Question: {question}

Output one query per line."""
)

generate_queries = (
    prompt_multi
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
)

# RAG-Fusion via Reciprocal Rank Fusion
def reciprocal_rank_fusion(results: list[list[Document]], k: int = 60):
    fused_scores = {}
    doc_map = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = json.dumps(doc.model_dump(), sort_keys=True)

            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
                doc_map[doc_str] = doc

            fused_scores[doc_str] += 1 / (rank + 1 + k)

    reranked_docs = sorted(
        doc_map.keys(),
        key=lambda x: fused_scores[x],
        reverse=True
    )

    return [doc_map[doc_str] for doc_str in reranked_docs]

# Multi-query + RAG-Fusion retrieval chain
retrieval_chain = generate_queries | retriever.map() | reciprocal_rank_fusion

# Run retrieval
retrieved_multi = retrieval_chain.invoke({"question": query})

print(f"RAG-Fusion retrieved {len(retrieved_multi)} fused chunks:\\n")
for i, doc in enumerate(retrieved_multi[:5]):   # top 5 after fusion
    source = doc.metadata.get("source", "unknown").split("/")[-1]
    print(f"[{i+1}] Source: {source}")
    print(doc.page_content[:300])
    print()

In [ ]:
from operator import itemgetter

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

answer_prompt = ChatPromptTemplate.from_template(
    """Answer the following question based only on this context:

{context}

Question: {question}
"""
)

final_rag_chain = (
    {
        "context": retrieval_chain | format_docs,
        "question": itemgetter("question")
    }
    | answer_prompt
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

answer = final_rag_chain.invoke({"question": query})
print("Final answer:\n")
print(answer)

### **Testing the Chatbot**

In [ ]:
demo_query = "What are the career outcomes of a UChicago MS-ADS student?"

answer = final_rag_chain.invoke({"question": demo_query})
retrieved_docs = retrieval_chain.invoke({"question": demo_query})[:5]

sources = []
for d in retrieved_docs:
    src = d.metadata["source"].split("/")[-1]
    if src not in sources:
        sources.append(src)

print("Question:", demo_query)
print()
print("Answer:", answer)
print()
print("Sources:", ", ".join(sources))

In [ ]:
queries = [
    "What are the career outcomes of a UChicago MS-ADS student?",
    "What is the average salary of a graduate in the program?",
    "Is the GRE or GMAT required?",
    "Do I need prior coding work experience to join this program?",
    "Does the online degree say 'online' on the diploma?",
    "How long is the program?",
    "Is there any career support provided by the program?",
    "What are the main core courses and electives?",
    "What Companies do graduates end up working for?",
    "What do i need to submit in my application?",
    "Is this part of the physical sciences department?",
    "What makes this program special?"
]

for i, q in enumerate(queries, 1):
    retrieved_docs = retrieval_chain.invoke({"question": q})[:5]
    answer = final_rag_chain.invoke({"question": q})

    sources = []
    for d in retrieved_docs:
        src = d.metadata.get("source", "unknown").split("/")[-1]
        if src not in sources:
            sources.append(src)

    print(f"Question {i} → {q}")
    print(f"Answer → {answer}")
    print(f"Source(s) → {', '.join(sources)}")
    print()

### **Creation of Streamlit End User Friendly Interface**

import streamlit as st

st.title("UChicago MS-ADS Chatbot")

query = st.text_input("Ask a question:")

if query:
    retrieved_docs = retrieval_chain.invoke({"question": query})
    context = format_docs(retrieved_docs)
    answer = answer_chain.invoke({"context": context, "question": query})

    st.write("Answer:")
    st.write(answer)